In [1]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from datetime import datetime, timezone

import models
import data_loader
import base
import meta
import plot
import evaluation
from trainers import ANN, LSTM, MetaModel

In [2]:
TICKER      = "AAPL"      # Change to any ticker you want
START_DATE  = "2013-01-01"
END_DATE    = datetime.now(timezone.utc).strftime("%Y-%m-%d")
WINDOW_SIZE = 1          # How many past days the model sees
BATCH_SIZE  = 16
N_SPLITS    = 5           # TimeSeriesSplit folds; we use the last one
EPOCHS = 500


OUTPUT_DIR  = "outputs"
EVALUATION_DIR = "evaluation"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(EVALUATION_DIR, exist_ok=True)

### Data Loader

In [3]:
# Downloading and building features from data
df = data_loader.download_data(TICKER, START_DATE, END_DATE)
data = data_loader.build_features(df)
n_features = data.shape[1]
scaled, f_scaler, c_scaler = data_loader.normalize(data)

# Creatong sequences and splitting dataset for training
X, y = data_loader.create_sequences(scaled, WINDOW_SIZE)
X_train, X_test, y_train, y_test = data_loader.split_data(X, y)

# Splitting into train and test datasets
train_loader = DataLoader(
    models.StockDataset(X_train, y_train), 
    batch_size=BATCH_SIZE, shuffle=True
    )
test_loader  = DataLoader(
    models.StockDataset(X_test,  y_test), 
    batch_size=BATCH_SIZE, shuffle=False
    )

[*********************100%***********************]  1 of 1 completed

Downloaded 3370 rows for AAPL
Train: 2808 samples | Test: 561 samples


### Models Loader

In [4]:
lstm_model = LSTM(n_features=n_features)
ann_model  = ANN(n_features=n_features, seq_len=WINDOW_SIZE)
meta_model = MetaModel()

params = (sum(p.numel() for p in lstm_model.parameters()) +
                sum(p.numel() for p in ann_model.parameters()) +
                sum(p.numel() for p in meta_model.parameters()))

In [5]:
params

130405

### Training MetaModel Process

In [6]:
# LSTM and ANN model training
lstm_model, lstm_tr_loss, lstm_val_loss = base.train(
    lstm_model, train_loader, test_loader, name="LSTM", epochs=EPOCHS
)
ann_model, ann_tr_loss, ann_val_loss = base.train(
    ann_model, train_loader, test_loader, name="ANN", epochs=EPOCHS
)

# Get predictions from each model
lstm_tr_preds, ann_tr_preds, y_tr_meta = base.predict(
    lstm_model, ann_model, train_loader
)
lstm_val_preds, ann_val_preds, y_val_meta = base.predict(
    lstm_model, ann_model, test_loader
)


Training LSTM on cpu
  Epoch  10/500 | Train MSE: 0.000352 | Val MSE: 0.000535
  Epoch  20/500 | Train MSE: 0.000275 | Val MSE: 0.000755
  Epoch  30/500 | Train MSE: 0.000244 | Val MSE: 0.000425
  Epoch  40/500 | Train MSE: 0.000231 | Val MSE: 0.000398
  Epoch  50/500 | Train MSE: 0.000231 | Val MSE: 0.000466
  Epoch  60/500 | Train MSE: 0.000238 | Val MSE: 0.000379
  Epoch  70/500 | Train MSE: 0.000252 | Val MSE: 0.000337
  Epoch  80/500 | Train MSE: 0.000246 | Val MSE: 0.000376
  Epoch  90/500 | Train MSE: 0.000228 | Val MSE: 0.000397
  Epoch 100/500 | Train MSE: 0.000242 | Val MSE: 0.000390
  Epoch 110/500 | Train MSE: 0.000222 | Val MSE: 0.000388
  Epoch 120/500 | Train MSE: 0.000238 | Val MSE: 0.000387
  Epoch 130/500 | Train MSE: 0.000241 | Val MSE: 0.000387
  Epoch 140/500 | Train MSE: 0.000222 | Val MSE: 0.000386
  Epoch 150/500 | Train MSE: 0.000235 | Val MSE: 0.000386
  Epoch 160/500 | Train MSE: 0.000234 | Val MSE: 0.000385
  Epoch 170/500 | Train MSE: 0.000232 | Val MSE: 0

In [7]:
# Using previous LSTM and ANN predicted results to train the meta-model
meta_model, meta_tr_loss, meta_val_loss = meta.train(
        meta_model,
        lstm_tr_preds, ann_tr_preds, y_tr_meta,
        lstm_val_preds, ann_val_preds, y_val_meta,
        epochs=EPOCHS
    )

Training Meta-Model (Stacking Layer)
  Epoch  10/500 | Train MSE: 0.013655 | Val MSE: 0.097508
  Epoch  20/500 | Train MSE: 0.002605 | Val MSE: 0.019020
  Epoch  30/500 | Train MSE: 0.000808 | Val MSE: 0.006655
  Epoch  40/500 | Train MSE: 0.000198 | Val MSE: 0.002288
  Epoch  50/500 | Train MSE: 0.000069 | Val MSE: 0.001001
  Epoch  60/500 | Train MSE: 0.000054 | Val MSE: 0.000675
  Epoch  70/500 | Train MSE: 0.000053 | Val MSE: 0.000608
  Epoch  80/500 | Train MSE: 0.000053 | Val MSE: 0.000592
  Epoch  90/500 | Train MSE: 0.000053 | Val MSE: 0.000592
  Epoch 100/500 | Train MSE: 0.000053 | Val MSE: 0.000591
  Epoch 110/500 | Train MSE: 0.000053 | Val MSE: 0.000577
  Epoch 120/500 | Train MSE: 0.000053 | Val MSE: 0.000593
  Epoch 130/500 | Train MSE: 0.000053 | Val MSE: 0.000597
  Epoch 140/500 | Train MSE: 0.000053 | Val MSE: 0.000591
  Epoch 150/500 | Train MSE: 0.000053 | Val MSE: 0.000570
  Epoch 160/500 | Train MSE: 0.000053 | Val MSE: 0.000560
  Epoch 170/500 | Train MSE: 0.0000

### Evaluating all models

In [8]:
# Get scaled predictions
lstm_pred_s, y_true_s = evaluation.predict(lstm_model, test_loader)
ann_pred_s, _ = evaluation.predict(ann_model,  test_loader)
ens_pred_s, _ = evaluation.ensemble_predictions(
    lstm_model, ann_model, meta_model, test_loader
    )

In [9]:
# Inverse-transform to real $ values
y_true_r    = evaluation.inverse_scale(y_true_s,   c_scaler)
lstm_pred_r = evaluation.inverse_scale(lstm_pred_s, c_scaler)
ann_pred_r  = evaluation.inverse_scale(ann_pred_s,  c_scaler)
ens_pred_r  = evaluation.inverse_scale(ens_pred_s,  c_scaler)

In [10]:
# Compute and display metrics (real $ scale)
print("\n  ─── Test Set Performance (real $ scale) ───")
print(f"  {'Model':<25} {'R²':>8}  {'MAE':>8}  {'MSE':>10}  {'RMSE':>8}")
results = {}
results["LSTM"]     = evaluation.compute_metrics(y_true_r, lstm_pred_r, "LSTM")
results["ANN"]      = evaluation.compute_metrics(y_true_r, ann_pred_r,  "ANN")
results["LSTM+ANN"] = evaluation.compute_metrics(y_true_r, ens_pred_r,  "LSTM+ANN (Ensemble)")


  ─── Test Set Performance (real $ scale) ───
  Model                           R²       MAE         MSE      RMSE
  LSTM                      R²=+0.9752  MAE=3.7636  MSE=25.1966  RMSE=5.0196
  ANN                       R²=+0.7134  MAE=14.1476  MSE=291.1243  RMSE=17.0624
  LSTM+ANN (Ensemble)       R²=+0.9708  MAE=4.1616  MSE=29.6204  RMSE=5.4425


In [11]:
# Improvement summary
r2_ann = results["ANN"]["R2"]
r2_ens = results["LSTM+ANN"]["R2"]
if r2_ann > 0:
    improvement = (r2_ens - r2_ann) / abs(r2_ann) * 100
    print(f"\n  Ensemble R² improvement over ANN: {improvement:+.1f}%")


  Ensemble R² improvement over ANN: +36.1%


### Saving plots

In [12]:
plot.training_curves(
    {"LSTM": (lstm_tr_loss, lstm_val_loss),
        "ANN":  (ann_tr_loss,  ann_val_loss),
        "Meta": (meta_tr_loss, meta_val_loss)},
    save_path=f"{OUTPUT_DIR}/training_curves.png"
)
plot.predictions(
    y_true_r,
    {"LSTM": lstm_pred_r, "ANN": ann_pred_r, "LSTM+ANN": ens_pred_r},
    ticker=TICKER,
    save_path=f"{OUTPUT_DIR}/predictions.png"
)
plot.metrict_comparison(
    results,
    save_path=f"{OUTPUT_DIR}/metrics_comparison.png"
)

# Save model weights
torch.save(lstm_model.state_dict(), f"{OUTPUT_DIR}/lstm_weights.pt")
torch.save(ann_model.state_dict(),  f"{OUTPUT_DIR}/ann_weights.pt")
torch.save(meta_model.state_dict(), f"{OUTPUT_DIR}/meta_weights.pt")

print(f"  Weights saved to {OUTPUT_DIR}/")

print("\n" + "="*60)
print("  Training complete.")
print(f"  Best model: LSTM+ANN Ensemble")
print(f"  R² = {r2_ens:.4f} | RMSE = {results['LSTM+ANN']['RMSE']:.4f}")
print("="*60 + "\n")

  Saved: outputs/training_curves.png
  Saved: outputs/predictions.png
  Saved: outputs/metrics_comparison.png
  Weights saved to outputs/

  Training complete.
  Best model: LSTM+ANN Ensemble
  R² = 0.9708 | RMSE = 5.4425



In [16]:
lstm_pred_r

array([180.02148, 179.92973, 176.31381, 171.60443, 170.91606, 170.17487,
       171.05124, 173.28072, 174.43802, 172.75531, 174.2894 , 173.13435,
       173.78749, 175.65094, 178.29572, 171.5091 , 173.00752, 171.37254,
       169.65048, 171.5881 , 171.63542, 169.99147, 169.35054, 170.20862,
       168.32532, 169.55637, 168.83313, 170.0735 , 168.09355, 174.11821,
       174.76472, 172.37384, 169.77837, 169.57855, 166.68098, 164.35431,
       165.84573, 166.9956 , 168.72723, 169.4998 , 169.6136 , 172.83627,
       170.05376, 169.19025, 171.98244, 180.22475, 178.01544, 180.42134,
       181.70192, 182.87077, 179.55075, 183.3014 , 184.18732, 186.27977,
       186.3716 , 187.44969, 188.00414, 189.35112, 188.02249, 185.14995,
       188.1339 , 187.70357, 187.75409, 189.18045, 191.1343 , 192.01399,
       192.62466, 194.0802 , 193.03949, 195.12611, 191.51704, 202.97748,
       206.92279, 209.53748, 208.61281, 214.33104, 209.20847, 205.20778,
       202.94855, 204.90623, 204.43471, 208.46213, 

### RAG Retrieval

In [13]:
import rag

In [14]:
rag.query(TICKER)

[4/4] Building vector store (20 articles, 60 price bars)…


{'analysis_metadata': {'ticker': 'AAPL',
  'timestamp': '2026-05-28T11:03:15Z',
  'data_window': '2026-03-03 to 2026-05-27'},
 'fundamental_indicators': {'market_capitalization_billions': 4565.56,
  'pe_ratio': 37.59,
  'dividend_yield_percent': 35.0,
  'fifty_two_week_high': 313.26,
  'fifty_two_week_low': 195.07,
  'latest_quarterly_dividend': 0.27},
 'news_corpus_analysis': [{'headline': 'AAPL Stock: The Math Behind The Upside - Trefis',
   'event_category': 'general',
   'sentiment_label': 'NO',
   'sentiment_score': -0.2667,
   'reasoning_summary': 'TextBlob polarity=-0.267; category=general; source=news.google.com',
   'source_relevance': 70},
  {'headline': 'Prediction: Apple Stock Will Trade at This Price in Two Years - Yahoo Finance',
   'event_category': 'general',
   'sentiment_label': 'UNKNOWN',
   'sentiment_score': 0.0,
   'reasoning_summary': 'TextBlob polarity=0.000; category=general; source=news.google.com',
   'source_relevance': 50},
  {'headline': "What's Going On W